# Домашнє завдання. Тема 5: Градієнтний спуск для поліноміальної регресії

In [1]:
import numpy as np
from sklearn.preprocessing import PolynomialFeatures

## 1. Генерація даних

In [2]:
np.random.seed(42)
x1 = np.random.rand(100)
x2 = np.random.rand(100)

def polynomial(x1, x2):
    return 4*x1**2 + 5*x2**2 - 2*x1*x2 + 3*x1 - 6*x2

y = polynomial(x1, x2)

X = np.column_stack([x1, x2])
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

X shape: (100, 2)
y shape: (100,)


## 2. Генерація додаткових ознак (PolynomialFeatures)

In [3]:
poly = PolynomialFeatures(degree=2, include_bias=True)
X_poly = poly.fit_transform(X)

print(f"X_poly shape: {X_poly.shape}")
print(f"Назви ознак: {poly.get_feature_names_out()}")

X_poly shape: (100, 6)
Назви ознак: ['1' 'x0' 'x1' 'x0^2' 'x0 x1' 'x1^2']


## 3. Реалізація методів градієнтного спуску

In [4]:
def compute_gradient(X, y, w):
    m = len(y)
    predictions = X @ w
    error = predictions - y
    gradient = (2 / m) * (X.T @ error)
    return gradient

def compute_mse(X, y, w):
    predictions = X @ w
    return np.mean((predictions - y) ** 2)

In [5]:
def polynomial_regression_gradient_descent(X, y, lr=0.01, n_iter=1000):
    w = np.zeros(X.shape[1])
    for i in range(n_iter):
        gradient = compute_gradient(X, y, w)
        w = w - lr * gradient
    mse = compute_mse(X, y, w)
    return w, mse

In [6]:
def polynomial_regression_SGD(X, y, lr=0.01, n_iter=1000):
    m = len(y)
    w = np.zeros(X.shape[1])
    for i in range(n_iter):
        idx = np.random.randint(m)
        xi = X[idx:idx+1]
        yi = y[idx:idx+1]
        gradient = compute_gradient(xi, yi, w)
        w = w - lr * gradient
    mse = compute_mse(X, y, w)
    return w, mse

In [7]:
def polynomial_regression_rmsprop(X, y, lr=0.01, n_iter=1000, beta=0.9, epsilon=1e-8):
    w = np.zeros(X.shape[1])
    s = np.zeros(X.shape[1])
    for i in range(n_iter):
        gradient = compute_gradient(X, y, w)
        s = beta * s + (1 - beta) * gradient ** 2
        w = w - lr * gradient / (np.sqrt(s) + epsilon)
    mse = compute_mse(X, y, w)
    return w, mse

In [8]:
def polynomial_regression_adam(X, y, lr=0.01, n_iter=1000, beta1=0.9, beta2=0.999, epsilon=1e-8):
    w = np.zeros(X.shape[1])
    m_vec = np.zeros(X.shape[1])
    v_vec = np.zeros(X.shape[1])
    for i in range(1, n_iter + 1):
        gradient = compute_gradient(X, y, w)
        m_vec = beta1 * m_vec + (1 - beta1) * gradient
        v_vec = beta2 * v_vec + (1 - beta2) * gradient ** 2
        m_hat = m_vec / (1 - beta1 ** i)
        v_hat = v_vec / (1 - beta2 ** i)
        w = w - lr * m_hat / (np.sqrt(v_hat) + epsilon)
    mse = compute_mse(X, y, w)
    return w, mse

In [9]:
def polynomial_regression_nadam(X, y, lr=0.01, n_iter=1000, beta1=0.9, beta2=0.999, epsilon=1e-8):
    w = np.zeros(X.shape[1])
    m_vec = np.zeros(X.shape[1])
    v_vec = np.zeros(X.shape[1])
    for i in range(1, n_iter + 1):
        gradient = compute_gradient(X, y, w)
        m_vec = beta1 * m_vec + (1 - beta1) * gradient
        v_vec = beta2 * v_vec + (1 - beta2) * gradient ** 2
        m_hat = m_vec / (1 - beta1 ** i)
        v_hat = v_vec / (1 - beta2 ** i)
        w = w - lr * (beta1 * m_hat + (1 - beta1) * gradient / (1 - beta1 ** i)) / (np.sqrt(v_hat) + epsilon)
    mse = compute_mse(X, y, w)
    return w, mse

## 4. Обчислення часу роботи (%timeit)

In [10]:
print("Gradient Descent:")
%timeit polynomial_regression_gradient_descent(X_poly, y, lr=0.01, n_iter=1000)

Gradient Descent:
30.1 ms ± 11.2 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [11]:
print("SGD:")
%timeit polynomial_regression_SGD(X_poly, y, lr=0.01, n_iter=1000)

SGD:
35.7 ms ± 12.6 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [12]:
print("RMSProp:")
%timeit polynomial_regression_rmsprop(X_poly, y, lr=0.01, n_iter=1000)

RMSProp:
19.7 ms ± 6.15 ms per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [13]:
print("Adam:")
%timeit polynomial_regression_adam(X_poly, y, lr=0.01, n_iter=1000)

Adam:
24.8 ms ± 7.11 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [14]:
print("Nadam:")
%timeit polynomial_regression_nadam(X_poly, y, lr=0.01, n_iter=1000)

Nadam:
The slowest run took 4.44 times longer than the fastest. This could mean that an intermediate result is being cached.
44.6 ms ± 25.1 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


## 5. Підбір оптимальної кількості ітерацій

In [15]:
methods = {
    'GD': polynomial_regression_gradient_descent,
    'SGD': polynomial_regression_SGD,
    'RMSProp': polynomial_regression_rmsprop,
    'Adam': polynomial_regression_adam,
    'Nadam': polynomial_regression_nadam
}

iterations = [100, 500, 1000, 2000, 5000, 10000]

print(f"{'Метод':<10}", end="")
for n in iterations:
    print(f"{n:>10}", end="")
print()
print("-" * 70)

for name, method in methods.items():
    print(f"{name:<10}", end="")
    for n in iterations:
        _, mse = method(X_poly, y, lr=0.01, n_iter=n)
        print(f"{mse:>10.6f}", end="")
    print()

Метод            100       500      1000      2000      5000     10000
----------------------------------------------------------------------
GD          1.723876  0.313510  0.203326  0.171074  0.111638  0.059199
SGD         1.928653  0.344531  0.206612  0.171746  0.120712  0.060241
RMSProp     1.286267  0.051995  0.002044  0.000234  0.000227  0.000227
Adam        1.623097  0.256374  0.116821  0.008991  0.000003  0.000000
Nadam       1.568986  0.251896  0.111455  0.007640  0.000003  0.000003


## 6. Висновок

1. **Gradient Descent (GD)** - базовий метод, стабільно збігається, але потребує більше ітерацій для досягнення низького MSE.

2. **SGD** - найшвидший за часом на одну ітерацію (оновлення по одному зразку), але через стохастичність потребує більше ітерацій і має нестабільну збіжність.

3. **RMSProp** - адаптивний метод, автоматично підлаштовує learning rate для кожного параметра. Збігається швидше за GD та SGD.

4. **Adam** - поєднує переваги momentum та RMSProp. Забезпечує швидку та стабільну збіжність, потребує менше ітерацій для досягнення низького MSE.

5. **Nadam** - модифікація Adam з Nesterov momentum. Показує схожу або дещо кращу збіжність порівняно з Adam.

Адаптивні методи (RMSProp, Adam, Nadam) досягають низького MSE за меншу кількість ітерацій, ніж GD та SGD, що робить їх більш ефективними для задач оптимізації.